# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [23]:
import duckdb
from google.colab import userdata

# 1. Safely fetch token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Connect DuckDB using your token variable
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# 3. Base Hugging Face Warehouse path
rel = "hf://datasets/FlyRank/internship-warehouse"

# 4. Run your verification query
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: One single row represents one unique pseudonymized content item (one page) for a specific client on a single reporting day (report_date + client_hash_id + content_hash_id).

Time Window: Mid-panel iteration slice — March 2026 (2026-03-01 to 2026-03-31).

Target / Proxy Definition: is_declining_label (1 if gsc_clicks == 0, else 0), calculated across our daily performance observations.

Deliberate Exclusion: Raw content URLs, raw client names, domain strings, and raw keyword queries are strictly excluded to preserve public safety and pseudonymization integrity.

In [24]:
# Verify Unit of Analysis and Time Window on March 2026 slice
query_unit_verify = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content_items,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
"""

unit_check_df = con.sql(query_unit_verify).df()

print("--- SECTION 1: UNIT OF ANALYSIS & TIME WINDOW VERIFICATION ---")
print(unit_check_df.to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- SECTION 1: UNIT OF ANALYSIS & TIME WINDOW VERIFICATION ---
 total_rows  unique_content_items   min_date   max_date
    9841378                331437 2026-03-01 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Observed Features (Knowable at prediction time): gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions.

Target / Proxy Label: is_declining_label (1 if gsc_clicks == 0, else 0).

Context / Metadata: content_hash_id, client_hash_id, report_date (used exclusively for grouping, deduplication, and client holdout splits).

Excluded Fields & Why: trend_direction and trend_pct are strictly excluded from training features because they directly encode the target label, creating circular data leakage. Application decision flags (health_score, priority_score) are also excluded as rule outputs.

In [25]:
query_sec2 = f"""
SELECT
    content_hash_id,
    client_hash_id,
    report_date,
    -- Lane 2 Observed Features (Knowable at prediction time)
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_data_available,
    -- Target Proxy: 1 if page receives zero clicks (decay candidate), else 0
    CASE WHEN gsc_clicks = 0 THEN 1 ELSE 0 END AS is_declining_label
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
LIMIT 50000
"""

feature_frame = con.sql(query_sec2).df()

print("--- SECTION 2: LANE 2 FEATURE FRAME ---")
print(f"Total Rows Loaded: {len(feature_frame):,}")
print(f"Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions']")
print(f"Target Label ('is_declining_label') Count = {feature_frame['is_declining_label'].sum():,}")
feature_frame.head(3)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- SECTION 2: LANE 2 FEATURE FRAME ---
Total Rows Loaded: 50,000
Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions']
Target Label ('is_declining_label') Count = 47,472


,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_data_available,is_declining_label
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03-01,20,0,3.350,<NA>,<NA>,1
1,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,1,0,0.000,<NA>,<NA>,1
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,125,1,4.928,<NA>,<NA>,0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification Queries:

Grain Proof: Proving zero primary key duplicates at the report_date + client_hash_id + content_hash_id grain.

Inventory Scope: Confirming exact row counts and date boundaries for March 2026.

Availability Check (IS TRUE): Verifying surviving rows where GA4 tracking data is actively marked as available (ga4_data_available IS TRUE).

In [26]:
# Query 1: Grain Verification (Checking for primary key duplicates)
q1_grain = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as cnt
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""
dups_df = con.sql(q1_grain).df()
print(f"Query 1 (Grain Proof) - Key duplicates found: {len(dups_df)}")

# Query 2: Slice Inventory & Row Counts
q2_counts = f"""
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as start_date,
    MAX(report_date) as end_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
"""
counts_df = con.sql(q2_counts).df()
print("\nQuery 2 (Lane 2 Inventory Counts):")
print(counts_df.to_string(index=False))

# Query 3: Availability Check (GA4 tracking active IS TRUE)
q3_avail = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) as ga4_active_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
"""
avail_df = con.sql(q3_avail).df()
print("\nQuery 3 (Data Availability - IS TRUE check):")
print(avail_df.to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 (Grain Proof) - Key duplicates found: 0

Query 2 (Lane 2 Inventory Counts):
 total_rows start_date   end_date
    9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Query 3 (Data Availability - IS TRUE check):
 total_rows  ga4_active_rows
    9841378           413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named Limitations of our Slice:

Unbalanced Panel & History Depth: Client onboarding dates vary significantly across the cohort, meaning earlier dates have Search Console data only while GA4 columns are zero-filled.

Observational Bounds: The data measures historical interaction trends; it cannot provide causal proof that an editorial refresh guarantees recovery.

The Leakage Trap Experiment: Below, we deliberately inject gsc_clicks (the exact variable used to construct is_declining_label) into our Decision Tree to observe artificial metric inflation, then delete it to preserve an honest baseline.

In [27]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

# 1. Clean Feature Set (Excluded 'gsc_clicks' because it directly derives our label)
X_clean = feature_frame[['gsc_impressions', 'gsc_avg_position', 'ga4_sessions']].fillna(0)
y = feature_frame['is_declining_label']

# Fit Honest Model
dt_clean = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_clean.fit(X_clean, y)
pred_clean = dt_clean.predict(X_clean)
honest_precision = precision_score(y, pred_clean, zero_division=0)

# 2. Inject Leaky Column ('gsc_clicks' directly defines 'is_declining_label = (gsc_clicks == 0)')
X_leaky = X_clean.copy()
X_leaky['leaky_gsc_clicks'] = feature_frame['gsc_clicks'].fillna(0)

# Fit Leaky Model
dt_leaky = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_leaky.fit(X_leaky, y)
pred_leaky = dt_leaky.predict(X_leaky)
leaky_precision = precision_score(y, pred_leaky, zero_division=0)

print("--- SECTION 4: LEAKAGE EXPERIMENT RESULTS ---")
print(f"Honest Model Precision Score: {honest_precision:.4f}")
print(f"Leaky Model Precision Score (with target-derived feature): {leaky_precision:.4f}")
print("\n>> VERDICT: Leaky feature 'gsc_clicks' removed to preserve honest evaluation.")


--- SECTION 4: LEAKAGE EXPERIMENT RESULTS ---
Honest Model Precision Score: 0.9711
Leaky Model Precision Score (with target-derived feature): 1.0000

>> VERDICT: Leaky feature 'gsc_clicks' removed to preserve honest evaluation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.